In [2]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

fatal: destination path 'kcw-analytics' already exists and is not an empty directory.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using folder: /content/drive/Shareddrives


In [3]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [4]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "MOBILE": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3003, 49)
Loaded: raw_syp_simas_sales_bills.csv -> (13148, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (28015, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38425, 38)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154252, 41)
Loaded: raw_hq_icmas_products.csv -> (114982, 94)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50307, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733701, 38)
Loaded: raw_hq_armas_receivable.csv -> (2232, 20)
Loaded: raw_hq_apmas_payable.csv -> (979, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276183, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13875, 32)


In [5]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/online/lazada"

In [6]:
import pandas as pd
import glob
import os

files = glob.glob(os.path.join(folder, "*.xlsx"))

df_lazada_raw = pd.concat(
    [
        pd.read_excel(
            f,
            sheet_name="Transaction Overview",
            header=0,        # explicitly say header is first row
            dtype={
                "Seller SKU": "string",
                "Transaction Number": "string",
                "Reference": "string",
            }
        )
        for f in files
    ],
    ignore_index=True
)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

In [7]:
df_lazada_raw.columns

Index(['Transaction Date', 'Transaction Type', 'Fee Name',
       'Transaction Number', 'Details', 'Seller SKU', 'Lazada SKU', 'Amount',
       'VAT in Amount', 'WHT Amount', 'WHT included in Amount', 'Statement',
       'Paid Status', 'Order No.', 'Order Item No.', 'Order Item Status',
       'Shipping Provider', 'Shipping Speed', 'Shipment Type', 'Reference',
       'Comment', 'PaymentRefId', 'ShortCode'],
      dtype='object')

In [65]:
df = df_lazada_raw.copy()

# ensure numeric
df["Amount"] = (
    pd.to_numeric(df["Amount"], errors="coerce")
)

# composite key
df["sku_txn_key"] = (
    df["Seller SKU"].astype("string") + "|" +
    df["Reference"].astype("string")
)

# aggregation
df_grouped = (
    df
    .groupby(["Seller SKU", "Reference"], dropna=False)
    .agg(
        amount_signed = ("Amount", "sum"),
        amount_positive = ("Amount", lambda s: s[s > 0].sum()),
        amount_negative = ("Amount", lambda s: s[s < 0].sum()),
        line_count = ("Amount", "size")
    )
    .reset_index()
)

In [66]:
df_grouped

,Seller SKU,Reference,amount_signed,amount_positive,amount_negative,line_count
0,05052418,1080221530476898,649.85,798.0,-148.15,6
1,05052418,1080221530576898,649.85,798.0,-148.15,6
2,07010524,1085363326136892,123.08,145.0,-21.92,4
3,08010644,1085698553112609,3208.74,3790.0,-581.26,5
4,08017381,1081633748889005,532.67,650.0,-117.33,5
...,...,...,...,...,...,...
406,35050126,1081976996526077,571.17,675.0,-103.83,5
407,35050150,1084565350216011,273.64,325.0,-51.36,4
408,5088996259-1709591764409-0,1071723465828335,1453.16,1640.0,-186.84,3
409,5968962800-1764129397314-0,1084478547990365,1212.63,1480.0,-267.37,5


In [53]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()

In [74]:
df_left = df_grouped.copy()

df_left["REF_TRIM_END"] = (
    df_left["Reference"]
    .astype("string")
    .str.strip()
    .str[:-1]
)

In [75]:
df_lookup = (
    df_simas[["PO", "BILLNO"]]
    .dropna(subset=["PO"])
    .drop_duplicates()
    .copy()
)

df_lookup["PO"] = df_lookup["PO"].astype("string").str.strip()
df_lookup["BILLNO"] = df_lookup["BILLNO"].astype("string").str.strip()

In [76]:
df_out = df_left.merge(
    df_lookup,
    left_on="REF_TRIM_END",
    right_on="PO",
    how="left"
)

In [77]:
df_out

,Seller SKU,Reference,amount_signed,amount_positive,amount_negative,line_count,REF_TRIM_END,PO,BILLNO
0,05052418,1080221530476898,649.85,798.0,-148.15,6,108022153047689,<NA>,<NA>
1,05052418,1080221530576898,649.85,798.0,-148.15,6,108022153057689,<NA>,<NA>
2,07010524,1085363326136892,123.08,145.0,-21.92,4,108536332613689,<NA>,<NA>
3,08010644,1085698553112609,3208.74,3790.0,-581.26,5,108569855311260,<NA>,<NA>
4,08017381,1081633748889005,532.67,650.0,-117.33,5,108163374888900,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...
408,5088996259-1709591764409-0,1071723465828335,1453.16,1640.0,-186.84,3,107172346582833,<NA>,<NA>
409,5968962800-1764129397314-0,1084478547990365,1212.63,1480.0,-267.37,5,108447854799036,<NA>,<NA>
410,<NA>,0,-125.00,0.0,-125.00,3,,,8K64-23961
411,<NA>,0,-125.00,0.0,-125.00,3,,,TA6606-030


In [106]:
key = "108447854799036"

df_found = df_simas[
    df_simas["PO"].astype("string").str.contains(key, na=False)
].copy()

In [107]:
df_found[["BILLNO","PO"]]

,BILLNO,PO


In [ ]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "03_curated",
    f"shopee.csv"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [ ]:
df_out[["BILLNO","deduction_pct"]].to_csv(folder, index=False, encoding="utf-8-sig")